# Linear cross-entropy benchmarking using Piquasso


Linear cross-entropy benchmarking, or LXEB, is a statistical test used to quantify whether samples produced by a photonic experiment are correlated with the ideal output distribution predicted by quantum theory. In photonic quantum advantage proposals, the experiment typically consists of preparing an input state $\rho$, evolving it through an $m$-mode interferometer $U$, and measuring the output photon-number pattern <cite data-footcite="lxeb"></cite>.

We denote by $p_{\rho,U}(\mathbf n)$ the ideal probability of observing an output occupation pattern $\mathbf n$. If the input state has fixed total photon number $n$, then the possible outcomes are occupation vectors $\mathbf n \in \mathcal S_{m,n}$, where $\mathcal S_{m,n}$ is the set of all $m$-mode photon-number configurations with total photon number $n$. If the physical input state does not have a sharply fixed photon number, one can instead define the distribution after postselecting on runs with total detected photon number $n$.

The actual experimental device samples from another distribution, which we denote by $q$. The purpose of LXEB is to test whether $q$ is correlated with the ideal distribution $p_{\rho,U}$.

### Definition of linear cross-entropy

The linear cross-entropy between the ideal distribution $p_{\rho,U}$ and the experimental distribution $q$ is
$$
\mathrm{LXE}(p_{\rho,U},q)
=
\mathbb E_{\mathbf n \sim q}\, p_{\rho,U}(\mathbf n)
=
\sum_{\mathbf n \in \mathcal S_{m,n}}
q(\mathbf n)\,p_{\rho,U}(\mathbf n).
$$
Operationally, this quantity asks the following question:

> If we draw samples from the experimental device, how large are their ideal probabilities?

Thus, LXEB rewards experimental samples that land on outcomes that are assigned high probability by the ideal quantum distribution. If $q$ is uncorrelated with $p_{\rho,U}$, then the LXE score is expected to be comparatively small. If $q = p_{\rho,U}$, meaning that the experiment perfectly samples from the ideal distribution, then

$$
\mathrm{LXE}(p_{\rho,U},p_{\rho,U})
=
\sum_{\mathbf n \in \mathcal S_{m,n}}
p_{\rho,U}(\mathbf n)^2.
$$

### Empirical LXEB score

In an experiment, we do not know the full distribution $q$ directly. Instead, we collect a finite set of samples
$$
\mathcal X = \{\mathbf n_1,\ldots,\mathbf n_s\},
\qquad
\mathbf n_i \sim q.
$$
The empirical LXEB score is
$$
\widehat{\mathrm{LXE}}(p_{\rho,U},\mathcal X)
=
\frac{1}{s}
\sum_{i=1}^s
p_{\rho,U}(\mathbf n_i).
$$
This estimator is unbiased, since
$$
\mathbb E_{\mathcal X \sim q}
\left[\widehat{\mathrm{LXE}}(p_{\rho,U},\mathcal X)\right]
=
\mathrm{LXE}(p_{\rho,U},q).
$$
Therefore, the empirical LXEB score estimates the correlation between the experimental samples and the ideal quantum probabilities.

### LXEB pass criterion

Given a tolerance parameter $\Delta \in [0,1)$, one can say that the sample set $\mathcal X$ passes the LXEB test if
$$
\frac{
\left|
\widehat{\mathrm{LXE}}(p_{\rho,U},\mathcal X)
-
\mathrm{LXE}_{\mathrm{ref}}^{(n)}(\rho)
\right|
}{
\mathrm{LXE}_{\mathrm{ref}}^{(n)}(\rho)
}
<
\Delta.
$$
In words, the empirical LXEB score must approximate the Haar-averaged ideal reference value $\mathrm{LXE}_{\mathrm{ref}}^{(n)}(\rho)$ up to relative error $\Delta$.
This criterion tests whether the experimental data have the same average ideal-probability weight as samples from the target distribution. A successful LXEB test therefore provides evidence that the device is producing samples correlated with the intended photonic quantum distribution.

## LXEB experiment for boson sampling with Piquasso

In this notebook, we simulate a small boson sampling experiment and evaluate its
linear cross-entropy benchmarking score.

The experiment has three parts:

1. Prepare an input Fock state with a fixed number of photons.
2. Apply a Haar-random interferometer and sample output photon-number patterns.
3. Compute the empirical LXEB score

$$
\widehat{\mathrm{LXE}}
=
\frac{1}{s}
\sum_{i=1}^s p_U(\mathbf n_i),
$$

where the samples $\mathbf n_i$ are produced by the simulated experiment and
$p_U(\mathbf n_i)$ is the ideal probability of observing that sample.

We compare the empirical score with the Haar-averaged reference value $\mathrm{LXE}_{\mathrm{ref}}^{(n)}(\rho)$ computed
using `piquasso.lxeb.lxe_ref_boson_sampling` in the following script.

In [8]:
import scipy
import piquasso as pq

from piquasso.lxeb import lxe_ref_boson_sampling

m = 32
n = 16

config = pq.Config(seed_sequence=42)
shots = 2000

U = scipy.stats.unitary_group.rvs(m, random_state=config.seed_sequence)

instructions = [pq.NumberState([1] * n + [0] * (m - n)), pq.Interferometer(U)]

state = pq.simulate(pq.Program(instructions), number_of_modes=m).state

samples = pq.simulate(
    pq.Program(instructions + [pq.ParticleNumberMeasurement()]),
    number_of_modes=m,
    shots=shots,
).samples

sum_ = 0.0

for sample in samples:
    sum_ += state.get_particle_detection_probability(sample)

lxe_estimate = sum_ / shots
print("Estimated value:", lxe_estimate)

lxe_reference = lxe_ref_boson_sampling(n, m)
print("Reference value:", lxe_reference)

relative_error = abs(lxe_reference - lxe_estimate) / lxe_reference
print("Relative error:", relative_error)

Estimated value: 2.3201076223045534e-12
Reference value: 2.3130728523936438e-12
Relative error: 0.0030413092711843275


Whenever there are losses in the experiment, the estimated LXE value dramatically differs from the reference value:

In [9]:
samples = pq.simulate(
    pq.Program(instructions + [pq.UniformLoss(0.85), pq.ParticleNumberMeasurement()]),
    number_of_modes=m,
    shots=shots,
).samples

sum_ = 0.0

for sample in samples:
    sum_ += state.get_particle_detection_probability(sample)

lossy_lxe_estimate = sum_ / shots
print("Lossy estimated value:", lossy_lxe_estimate)

relative_error = abs(lxe_reference - lossy_lxe_estimate) / lxe_reference
print("Relative error:", relative_error)

Lossy estimated value: 7.29166920647751e-15
Relative error: 0.9968476266543304


LXEB is useful because it reduces a complicated distributional comparison to a single scalar statistic. This is especially important in quantum advantage experiments, where the number of possible output patterns
$$
|\mathcal S_{m,n}| = \binom{m+n-1}{n}
$$
can be extremely large. However, several caveats are important:

1. LXEB is not a full certification of the output distribution.
   Passing the test shows correlation with the ideal probabilities, but it does not prove that $q = p_{\rho,U}$.

2. The ideal probabilities $p_{\rho,U}(\mathbf n_i)$ must be computed for the observed samples.
   These probabilities may involve matrix permanents or related quantities, which can be computationally expensive.

3. Finite-sample fluctuations matter.
   Even if the experiment samples from the ideal distribution, the empirical score fluctuates around its expectation. The sample size $s$ must therefore be large enough for the desired relative precision, but numerical evidence suggests that this procedure remains sample-efficient <cite data-footcite="lxeb"></cite>.

In summary, LXEB provides a practical way to benchmark photonic quantum advantage experiments by checking whether experimentally observed samples are concentrated on outputs that the ideal quantum model predicts to be likely.